# Tangible Grocery Cart Simulator — Gesture Recognition (Python side)

**3 Gestures (motion, equivalent to TUIO actions):**
| Gesture | Action | TUIO equivalent |
|---|---|---|
| Hand moves UP | Add to Cart | marker enters Cart Zone |
| Hand moves DOWN | Remove from Cart | marker leaves Cart Zone |
| Circle motion | Checkout | (separate confirm event) |

**Before running:** put your recorded videos in this structure next to this notebook:
```
Dataset/
  add_to_cart/
    add_to_cart_1.mp4
    add_to_cart_2.mp4
    add_to_cart_3.mp4
  remove_from_cart/
    remove_from_cart_1.mp4
    remove_from_cart_2.mp4
    remove_from_cart_3.mp4
  checkout/
    checkout_1.mp4
    checkout_2.mp4
    checkout_3.mp4
```
(keep 1 extra video per gesture OUTSIDE this folder, or named with `_test`, to use for testing later)


In [1]:
# Run once inside your cart-env (Anaconda Prompt):
# pip install mediapipe opencv-python dollarpy


In [2]:
import os
import cv2
import mediapipe as mp
from dollarpy import Recognizer, Template, Point

mp_hands = mp.solutions.holistic  # using Holistic so we can extend to pose later if needed
mp_drawing = mp.solutions.drawing_utils


## 1. getPoints() — extract the motion path from a video

We track the **right wrist** across frames (one point per frame).
That's enough to describe an "up", "down", or "circle" path — DollarPy compares
the overall shape/trajectory, not exact pixel values.


In [3]:
def getPoints(video_path, label=None):
    cap = cv2.VideoCapture(video_path)
    points = []

    with mp_hands.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = holistic.process(image)

            # Track whichever hand is visible (right or left)
            hand = results.right_hand_landmarks or results.left_hand_landmarks
            if hand:
                wrist = hand.landmark[0]  # landmark 0 = wrist
                points.append(Point(wrist.x, wrist.y, 1))  # strokeId=1 (single continuous stroke)

    cap.release()
    return points

## 2. Build templates automatically from the Dataset folder

Loops over `Dataset/<gesture_name>/*.mp4`, extracts points from each video,
and creates one DollarPy `Template` per video. The folder name IS the label.


In [4]:
def build_templates(dataset_path="Dataset"):
    templates = []

    for gesture_name in os.listdir(dataset_path):
        gesture_folder = os.path.join(dataset_path, gesture_name)
        if not os.path.isdir(gesture_folder):
            continue

        for video_file in os.listdir(gesture_folder):
            if not video_file.lower().endswith((".mp4", ".mov", ".avi")):
                continue

            video_path = os.path.join(gesture_folder, video_file)
            print(f"Processing {video_path} ...")

            points = getPoints(video_path, gesture_name)
            if len(points) < 2:
                print(f"  WARNING: no hand detected in {video_file}, skipping")
                continue

            templates.append(Template(gesture_name, points))

    print(f"\nBuilt {len(templates)} templates total.")
    return templates


In [5]:
templates = build_templates("Dataset")
recognizer = Recognizer(templates)


Processing Dataset\add_to_cart\add1.mp4 ...


c:\Users\HP\anaconda3\envs\HCI_Project\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing Dataset\add_to_cart\add2.mp4 ...
Processing Dataset\add_to_cart\add3.mp4 ...
Processing Dataset\checkout\check1.mp4 ...
Processing Dataset\checkout\check2.mp4 ...
Processing Dataset\checkout\check3.mp4 ...
Processing Dataset\remove_from_cart\remove_1.mp4 ...
Processing Dataset\remove_from_cart\remove_2.mp4 ...
Processing Dataset\remove_from_cart\remove_3.mp4 ...

Built 9 templates total.


## 3. Test on a held-out video (not used for training)


In [6]:
test_video = "Dataset_test/add_test.mp4"
test_points = getPoints(test_video)

result = recognizer.recognize(test_points)
print("Predicted gesture:", result[0])
print("Confidence score:", result[1])

Predicted gesture: add_to_cart
Confidence score: 0.9123511493684693


In [16]:
test_files = {
    "add_to_cart": "Dataset_test/add_test.mp4",
    "remove_from_cart": "Dataset_test/remove_test.mp4",
    "checkout": "Dataset_test/checkout_test.mp4",
}

for expected_label, path in test_files.items():
    p = getPoints(path)
    result = recognizer.recognize(p)
    correct = "✅" if result[0] == expected_label else "❌"
    print(f"{correct} Expected: {expected_label} | Predicted: {result[0]} | Score: {result[1]} | Points: {len(p)}")

✅ Expected: add_to_cart | Predicted: add_to_cart | Score: 0.9123511493684693 | Points: 172
✅ Expected: remove_from_cart | Predicted: remove_from_cart | Score: 0.8940282427392074 | Points: 233
✅ Expected: checkout | Predicted: checkout | Score: 0.7490618561802007 | Points: 327


In [12]:
points = getPoints("Dataset_test/checkout_test.mp4")
print("عدد النقاط:", len(points))
print("x range:", max(p.x for p in points) - min(p.x for p in points))
print("y range:", max(p.y for p in points) - min(p.y for p in points))

عدد النقاط: 228
x range: 0.4860712140798569
y range: 0.27287259697914124


In [14]:
for i in [1, 2, 3]:
    p = getPoints(f"Dataset/checkout/check{i}.mp4")
    result = recognizer.recognize(p)
    print(f"check{i}.mp4 → {result[0]} | score: {result[1]}")

check1.mp4 → checkout | score: 0.9999999999999999
check2.mp4 → checkout | score: 1.0
check3.mp4 → checkout | score: 0.9999999999999999


In [15]:
points = getPoints("Dataset_test/checkout_test.mp4")
print("عدد النقاط:", len(points))

result = recognizer.recognize(points)
print("Predicted gesture:", result[0])
print("Confidence score:", result[1])

عدد النقاط: 297
Predicted gesture: checkout
Confidence score: 0.7490618561802007


## 4. Live recognition from webcam

Records a short window of wrist motion, then classifies it.
Press **s** to start recording a gesture, **e** to end/classify it, **q** to quit.

This is a starting skeleton — next step will be sending the recognized gesture
to the C# TUIO app over a socket (same pattern as `serverPython.ipynb` from Lab 3).


In [8]:
import socket

def send_to_csharp(command):
    try:
        s = socket.socket()
        s.connect(("localhost", 6000))
        s.send(command.encode("utf-8"))
        s.close()
    except Exception as e:
        print("Could not reach C# app:", e)

In [21]:
def live_recognition_auto():
    cap = cv2.VideoCapture(0)
    live_points = []
    recording = False
    still_frames = 0

    MOTION_THRESHOLD = 0.01      # لو الحركة بين فريمين أكبر من كده، يبقى "بيتحرك"
    STILL_FRAMES_TO_STOP = 8     # كام فريم ثابت متتالي عشان نعتبر إنه وقف
    MIN_POINTS_TO_CLASSIFY = 15  # أقل عدد نقاط عشان نعتبرها حركة حقيقية

    prev_wrist = None

    try:
        with mp_hands.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
            print("Watching for hand movement... (press Interrupt/Stop to end)")
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frame = cv2.flip(frame, 1)
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(image)

                hand = results.right_hand_landmarks or results.left_hand_landmarks
                if hand:
                    wrist = hand.landmark[0]

                    if prev_wrist is not None:
                        dx = wrist.x - prev_wrist.x
                        dy = wrist.y - prev_wrist.y
                        movement = (dx ** 2 + dy ** 2) ** 0.5

                        if movement > MOTION_THRESHOLD:
                            recording = True
                            still_frames = 0
                            live_points.append(Point(wrist.x, wrist.y, 1))
                        elif recording:
                            still_frames += 1
                            live_points.append(Point(wrist.x, wrist.y, 1))

                            if still_frames >= STILL_FRAMES_TO_STOP:
                                print("Points collected:", len(live_points))
                                if len(live_points) >= MIN_POINTS_TO_CLASSIFY:
                                    result = recognizer.recognize(live_points)
                                    print("Detected gesture:", result[0], "| score:", result[1])
                                    if result[0] is not None:
                                        send_to_csharp(result[0])
                                recording = False
                                live_points = []
                                still_frames = 0

                    prev_wrist = wrist

    except KeyboardInterrupt:
        print("Stopped watching.")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Camera released.")

live_recognition_auto()

Watching for hand movement... (press Interrupt/Stop to end)
Points collected: 38
Detected gesture: add_to_cart | score: 0.389238128641627
Points collected: 31
Detected gesture: remove_from_cart | score: 0.3865457018133853
Points collected: 53
Detected gesture: remove_from_cart | score: 0.665552292203663
Points collected: 44
Detected gesture: add_to_cart | score: 0.2736718431408919
Points collected: 55
Detected gesture: checkout | score: 0.21637948079138658
Stopped watching.
Camera released.


In [10]:
cv2.destroyAllWindows()

In [25]:
import bluetooth

print("Scanning...")
devices = bluetooth.discover_devices(duration=8, lookup_names=True)

for addr, name in devices:
    print(f"{addr}  |  {name}")

    

Scanning...


In [23]:
import bluetooth
import socket
import time

MY_DEVICE_MAC = "FC:38:82:51:8E:6F"  # Infinix SMART Series zika

def send_to_csharp(command):
    try:
        s = socket.socket()
        s.connect(("localhost", 6000))
        s.send(command.encode("utf-8"))
        s.close()
    except Exception as e:
        print("Could not reach C# app:", e)

def watch_for_my_device():
    print("Watching for known device... (Interrupt/Stop to end)")
    already_sent = False

    try:
        while True:
            devices = bluetooth.discover_devices(duration=6, lookup_names=False)

            if MY_DEVICE_MAC in devices:
                if not already_sent:
                    print("Device found! Adding Milk to cart.")
                    send_to_csharp("add_milk")
                    already_sent = True
            else:
                already_sent = False  # reset so it can trigger again if it leaves and comes back

            time.sleep(2)
    except KeyboardInterrupt:
        print("Stopped watching.")

watch_for_my_device()

Watching for known device... (Interrupt/Stop to end)
Stopped watching.
